In [55]:
# ========================================
#   Install and Import Dependencies
# ========================================

!pip install torch torchvision matplotlib seaborn scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, models
from PIL import Image
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import time
import copy
from pathlib import Path
import requests
from zipfile import ZipFile
import random

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [56]:
# ========================================
# Dataset Setup Functions
# ========================================

def download_sample_dataset():
    """
    Download a sample dataset for demonstration.
    You can replace this with your own dataset loading function.
    """
    print("Setting up sample dataset structure...")

    # Create directory structure
    os.makedirs('dataset/train/cats', exist_ok=True)
    os.makedirs('dataset/train/dogs', exist_ok=True)
    os.makedirs('dataset/val/cats', exist_ok=True)
    os.makedirs('dataset/val/dogs', exist_ok=True)

    print("Dataset structure created!")
    print("Please upload your images to:")
    print("- dataset/train/cats/ (training cat images)")
    print("- dataset/train/dogs/ (training dog images)")
    print("- dataset/val/cats/ (validation cat images)")
    print("- dataset/val/dogs/ (validation dog images)")

    return 'dataset'

In [57]:
def download_flowers_dataset():
    """
    Download Flowers dataset from TensorFlow (RECOMMENDED FOR BEGINNERS)
    """
    import tensorflow as tf
    from pathlib import Path
    import shutil

    print("🌸 Downloading Flowers dataset...")

    # Download the dataset
    dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

    # Download and extract
    data_dir = tf.keras.utils.get_file('flower_photos',
                                       origin=dataset_url,
                                       untar=True,
                                       cache_dir='.')

    data_dir = Path(data_dir)

    # Get class names
    classes = [item.name for item in data_dir.iterdir() if item.is_dir() and not item.name.startswith('.')]

    print(f"✅ Flowers dataset downloaded!")
    print(f"Classes found: {classes}")
    print(f"Location: {data_dir}")

    # Convert to train/val split
    organized_dir = organize_flowers_dataset(data_dir, classes)

    return organized_dir, classes

In [58]:
def organize_flowers_dataset(data_dir, classes, split_ratio=0.8):
    """
    Organize flowers dataset into train/val split
    """
    import random
    import shutil
    from pathlib import Path

    print("📁 Organizing dataset into train/val split...")

    # Create organized directory structure
    organized_dir = Path('flowers_organized')
    for split in ['train', 'val']:
        for class_name in classes:
            (organized_dir / split / class_name).mkdir(parents=True, exist_ok=True)

    # Split each class into train/val
    for class_name in classes:
        class_path = data_dir / class_name
        if not class_path.exists():
            continue

        # Get all image files
        image_files = list(class_path.glob('*.jpg'))
        random.shuffle(image_files)

        # Calculate split index
        split_idx = int(len(image_files) * split_ratio)

        # Split files
        train_files = image_files[:split_idx]
        val_files = image_files[split_idx:]

        # Copy files to organized structure
        for img_file in train_files:
            shutil.copy2(img_file, organized_dir / 'train' / class_name / img_file.name)

        for img_file in val_files:
            shutil.copy2(img_file, organized_dir / 'val' / class_name / img_file.name)

        print(f"  {class_name}: {len(train_files)} train, {len(val_files)} val")

    print(f"✅ Dataset organized at: {organized_dir}")
    return str(organized_dir)

In [59]:
def download_cifar10_as_folders():
    """
    Download CIFAR-10 and save as image folders
    """
    import torchvision
    import torchvision.transforms as transforms
    from PIL import Image
    import numpy as np

    print("🚗 Downloading CIFAR-10 dataset...")

    # CIFAR-10 classes
    classes = ('plane', 'car', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck')

    # Create directories
    base_dir = Path('cifar10_organized')
    for split in ['train', 'val']:
        for class_name in classes:
            (base_dir / split / class_name).mkdir(parents=True, exist_ok=True)

    # Download CIFAR-10
    transform = transforms.Compose([transforms.ToPILImage()])

    trainset = torchvision.datasets.CIFAR10(root='./cifar10_raw', train=True,
                                            download=True, transform=None)
    testset = torchvision.datasets.CIFAR10(root='./cifar10_raw', train=False,
                                           download=True, transform=None)

    # Convert training set to folders
    print("Converting training set...")
    for i, (img, label) in enumerate(trainset):
        if i < 40000:  # Use first 40k for training
            img = Image.fromarray(img)
            class_name = classes[label]
            img.save(base_dir / 'train' / class_name / f'train_{i}.png')
        else:  # Use last 10k for validation
            img = Image.fromarray(img)
            class_name = classes[label]
            img.save(base_dir / 'val' / class_name / f'val_{i}.png')

        if i % 5000 == 0:
            print(f"  Processed {i}/50000 training images")

    print(f"✅ CIFAR-10 organized at: {base_dir}")
    return str(base_dir), classes

In [60]:
def download_cats_vs_dogs_kaggle():
    """
    Download Cats vs Dogs from Kaggle (requires Kaggle API)
    """
    try:
        # Install Kaggle API
        os.system('pip install kaggle')

        print("🐱🐶 Setting up Cats vs Dogs dataset...")
        print("📋 Please make sure you have:")
        print("1. Kaggle account")
        print("2. Kaggle API token (kaggle.json)")

        from google.colab import files

        # Check if kaggle.json already exists
        if not os.path.exists('/root/.kaggle/kaggle.json'):
            print("Please upload your kaggle.json file:")
            uploaded = files.upload()

            # Setup Kaggle credentials
            os.makedirs('/root/.kaggle', exist_ok=True)
            os.system('cp kaggle.json /root/.kaggle/')
            os.system('chmod 600 /root/.kaggle/kaggle.json')

        # Download dataset
        os.system('kaggle competitions download -c dogs-vs-cats')
        os.system('unzip -q dogs-vs-cats.zip')
        os.system('unzip -q train.zip')

        # Organize dataset
        organized_dir = organize_cats_dogs_dataset()

        return organized_dir, ['cats', 'dogs']

    except Exception as e:
        print(f"❌ Error downloading Cats vs Dogs: {e}")
        print("💡 Try using the Flowers dataset instead: download_flowers_dataset()")
        return None, None

In [61]:
def organize_cats_dogs_dataset():
    """
    Organize cats vs dogs into proper train/val structure
    """
    import random
    from pathlib import Path
    import shutil

    print("📁 Organizing cats vs dogs dataset...")

    # Create directories
    base_dir = Path('cats_dogs_organized')
    for split in ['train', 'val']:
        for animal in ['cats', 'dogs']:
            (base_dir / split / animal).mkdir(parents=True, exist_ok=True)

    # Get all training images
    train_files = list(Path('train').glob('*.jpg'))

    # Separate cats and dogs
    cat_files = [f for f in train_files if 'cat' in f.name]
    dog_files = [f for f in train_files if 'dog' in f.name]

    # Split and organize
    def split_and_move(files, animal_name):
        random.shuffle(files)
        split_idx = int(0.8 * len(files))

        train_files = files[:split_idx]
        val_files = files[split_idx:]

        # Copy training files
        for f in train_files:
            shutil.copy2(f, base_dir / 'train' / animal_name / f.name)

        # Copy validation files
        for f in val_files:
            shutil.copy2(f, base_dir / 'val' / animal_name / f.name)

        print(f"  {animal_name}: {len(train_files)} train, {len(val_files)} val")

    split_and_move(cat_files, 'cats')
    split_and_move(dog_files, 'dogs')

    print(f"✅ Cats vs Dogs organized at: {base_dir}")
    return str(base_dir)

In [62]:
def auto_download_dataset(dataset_choice='flowers'):
    """
    Automatically download and setup dataset based on choice

    Args:
        dataset_choice: 'flowers', 'cifar10', 'cats_dogs', or 'manual'
    """

    if dataset_choice.lower() == 'flowers':
        print("🌸 Setting up Flowers dataset (RECOMMENDED)...")
        return download_flowers_dataset()

    elif dataset_choice.lower() == 'cifar10':
        print("🚗 Setting up CIFAR-10 dataset...")
        return download_cifar10_as_folders()

    elif dataset_choice.lower() == 'cats_dogs':
        print("🐱🐶 Setting up Cats vs Dogs dataset...")
        return download_cats_vs_dogs_kaggle()

    elif dataset_choice.lower() == 'manual':
        print("📁 Setting up manual dataset structure...")
        return download_sample_dataset(), ['class1', 'class2']

    else:
        print("❌ Invalid choice. Using Flowers dataset as default...")
        return download_flowers_dataset()

In [63]:
def create_data_transforms():
    """Create data transforms for training and validation"""

    # Data augmentation and normalization for training
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Just normalization for validation
    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    return train_transforms, val_transforms

In [64]:
def load_datasets(data_dir, train_transforms, val_transforms, batch_size=32):
    """Load training and validation datasets"""

    # Create datasets
    train_dataset = torchvision.datasets.ImageFolder(
        root=os.path.join(data_dir, 'train'),
        transform=train_transforms
    )

    val_dataset = torchvision.datasets.ImageFolder(
        root=os.path.join(data_dir, 'val'),
        transform=val_transforms
    )

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2
    )

    # Get class names
    class_names = train_dataset.classes

    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Classes: {class_names}")
    print(f"Number of classes: {len(class_names)}")

    return train_loader, val_loader, class_names

In [65]:
# ========================================
# Model Creation Functions
# ========================================

def create_resnet_model(num_classes, pretrained=True, freeze_features=True):
    """
    Create a ResNet model for transfer learning

    Args:
        num_classes: Number of output classes
        pretrained: Whether to use pretrained weights
        freeze_features: Whether to freeze feature extraction layers
    """

    # Load pretrained ResNet-18
    model = models.resnet18(pretrained=pretrained)

    # Freeze feature extraction layers if specified
    if freeze_features:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the final layer
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    return model

In [66]:
def create_vgg_model(num_classes, pretrained=True, freeze_features=True):
    """
    Create a VGG model for transfer learning

    Args:
        num_classes: Number of output classes
        pretrained: Whether to use pretrained weights
        freeze_features: Whether to freeze feature extraction layers
    """

    # Load pretrained VGG-16
    model = models.vgg16(pretrained=pretrained)

    # Freeze feature extraction layers if specified
    if freeze_features:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace the final classifier
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, num_classes)

    return model

In [67]:
def create_custom_cnn_model(num_classes):
    """
    Create a simple custom CNN for comparison
    """

    model = nn.Sequential(
        # First block
        nn.Conv2d(3, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # Second block
        nn.Conv2d(32, 64, 3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # Third block
        nn.Conv2d(64, 128, 3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # Fourth block
        nn.Conv2d(128, 256, 3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((7, 7)),

        # Classifier
        nn.Flatten(),
        nn.Dropout(0.5),
        nn.Linear(256 * 7 * 7, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )

    return model

In [68]:
# ========================================
# Training Functions
# ========================================

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=25, device=device):
    """
    Train the model with validation
    """

    since = time.time()

    # Keep track of training history
    train_acc_history = []
    val_acc_history = []
    train_loss_history = []
    val_loss_history = []

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimize only in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler:
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Keep track of history
            if phase == 'train':
                train_acc_history.append(epoch_acc.cpu())
                train_loss_history.append(epoch_loss)
            else:
                val_acc_history.append(epoch_acc.cpu())
                val_loss_history.append(epoch_loss)

            # Deep copy the model if it's the best so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)

    history = {
        'train_acc': train_acc_history,
        'val_acc': val_acc_history,
        'train_loss': train_loss_history,
        'val_loss': val_loss_history
    }

    return model, history

In [69]:
def fine_tune_model(model, train_loader, val_loader, num_epochs=10,
                   learning_rate=1e-4, device=device):
    """
    Fine-tune the entire model with a lower learning rate
    """

    print("Starting fine-tuning phase...")

    # Unfreeze all parameters
    for param in model.parameters():
        param.requires_grad = True

    # Use a smaller learning rate for fine-tuning
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    model, history = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        num_epochs=num_epochs, device=device
    )

    return model, history


In [70]:
# ========================================
# Evaluation Functions
# ========================================

def evaluate_model(model, test_loader, class_names, device=device):
    """
    Evaluate the model and return predictions and metrics
    """

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate accuracy
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

    # Generate classification report
    report = classification_report(all_labels, all_preds,
                                 target_names=class_names,
                                 output_dict=True)

    print(f"Test Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    return all_preds, all_labels, accuracy, report

In [71]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Plot confusion matrix
    """

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [72]:
def plot_training_history(history, title="Training History"):
    """
    Plot training and validation accuracy/loss
    """

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot accuracy
    ax1.plot(history['train_acc'], label='Training Accuracy')
    ax1.plot(history['val_acc'], label='Validation Accuracy')
    ax1.set_title(f'{title} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    # Plot loss
    ax2.plot(history['train_loss'], label='Training Loss')
    ax2.plot(history['val_loss'], label='Validation Loss')
    ax2.set_title(f'{title} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [73]:
def visualize_predictions(model, test_loader, class_names, num_images=8, device=device):
    """
    Visualize model predictions on test images
    """

    model.eval()

    # Get a batch of test images
    dataiter = iter(test_loader)
    images, labels = next(dataiter)

    # Make predictions
    with torch.no_grad():
        outputs = model(images.to(device))
        _, predictions = torch.max(outputs, 1)

    # Plot images with predictions
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.ravel()

    for i in range(min(num_images, len(images))):
        # Denormalize image for display
        img = images[i].clone()
        img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        img = torch.clamp(img, 0, 1)

        # Convert to numpy and transpose
        img = img.numpy().transpose((1, 2, 0))

        axes[i].imshow(img)
        axes[i].set_title(f'True: {class_names[labels[i]]}\n'
                         f'Pred: {class_names[predictions[i]]}')
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [74]:
# ========================================
# Model Comparison Functions
# ========================================

def compare_models(train_loader, val_loader, class_names, num_epochs=15):
    """
    Compare different models: ResNet, VGG, and Custom CNN
    """

    results = {}

    # Define models to compare
    models_to_test = {
        'ResNet-18': create_resnet_model(len(class_names)),
        'VGG-16': create_vgg_model(len(class_names)),
        'Custom CNN': create_custom_cnn_model(len(class_names))
    }

    for model_name, model in models_to_test.items():
        print(f"\n{'='*50}")
        print(f"Training {model_name}")
        print(f"{'='*50}")

        model = model.to(device)

        # Define optimizer and criterion
        if 'Custom' in model_name:
            optimizer = optim.Adam(model.parameters(), lr=0.001)
        else:
            # For transfer learning, only train the classifier initially
            optimizer = optim.Adam(model.parameters(), lr=0.001)

        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
        criterion = nn.CrossEntropyLoss()

        # Train the model
        trained_model, history = train_model(
            model, train_loader, val_loader, criterion, optimizer, scheduler,
            num_epochs=num_epochs, device=device
        )

        # Evaluate the model
        preds, labels, accuracy, report = evaluate_model(
            trained_model, val_loader, class_names, device=device
        )

        # Store results
        results[model_name] = {
            'model': trained_model,
            'history': history,
            'accuracy': accuracy,
            'report': report
        }

        # Plot training history
        plot_training_history(history, f"{model_name}")

    return results

In [75]:
def plot_model_comparison(results):
    """
    Plot comparison of different models
    """

    model_names = list(results.keys())
    accuracies = [results[name]['accuracy'] for name in model_names]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(model_names, accuracies, color=['blue', 'green', 'red'])
    plt.title('Model Comparison - Test Accuracy')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)

    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.3f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

In [76]:
# ========================================
# Save and Load Functions
# ========================================

def save_model(model, filepath, class_names):
    """
    Save the trained model
    """

    torch.save({
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
    }, filepath)

    print(f"Model saved to {filepath}")

In [77]:
def load_model(filepath, model_type='resnet', num_classes=2):
    """
    Load a saved model
    """

    # Create model architecture
    if model_type == 'resnet':
        model = create_resnet_model(num_classes, pretrained=False)
    elif model_type == 'vgg':
        model = create_vgg_model(num_classes, pretrained=False)
    else:
        model = create_custom_cnn_model(num_classes)

    # Load saved state
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    class_names = checkpoint['class_names']

    model = model.to(device)
    print(f"Model loaded from {filepath}")

    return model, class_names

In [78]:
# ========================================
# Main Execution Pipeline
# ========================================

def run_transfer_learning_pipeline(dataset_choice='flowers'):
    """
    Complete pipeline for transfer learning project

    Args:
        dataset_choice: 'flowers', 'cifar10', 'cats_dogs', or 'manual'
    """

    print("🚀 Starting Transfer Learning Pipeline")
    print("="*50)

    # Step 1: Setup dataset
    print("📁 Setting up dataset...")
    data_dir, class_names = auto_download_dataset(dataset_choice)

    if data_dir is None:
        print("❌ Failed to setup dataset. Exiting...")
        return None, None

    # Step 2: Create data transforms and loaders
    print("🔄 Creating data transforms...")
    train_transforms, val_transforms = create_data_transforms()

    try:
        train_loader, val_loader, class_names = load_datasets(
            data_dir, train_transforms, val_transforms, batch_size=32
        )
    except Exception as e:
        print(f"❌ Could not load dataset: {e}")
        print("💡 Make sure the dataset was downloaded correctly.")
        return None, None

    # Step 3: Compare different models
    print("🏁 Comparing different models...")
    results = compare_models(train_loader, val_loader, class_names, num_epochs=10)

    # Step 4: Plot comparison
    plot_model_comparison(results)

    # Step 5: Fine-tune the best model
    best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
    best_model = results[best_model_name]['model']

    print(f"🏆 Best model: {best_model_name}")
    print("🔧 Fine-tuning the best model...")

    fine_tuned_model, ft_history = fine_tune_model(
        best_model, train_loader, val_loader, num_epochs=5
    )

    plot_training_history(ft_history, f"{best_model_name} - Fine-tuned")

    # Step 6: Final evaluation
    print("📊 Final evaluation...")
    preds, labels, accuracy, report = evaluate_model(
        fine_tuned_model, val_loader, class_names
    )

    plot_confusion_matrix(labels, preds, class_names)
    visualize_predictions(fine_tuned_model, val_loader, class_names)

    # Step 7: Save the best model
    save_model(fine_tuned_model, f'best_{best_model_name.lower().replace("-", "_")}_model.pth', class_names)

    print("✅ Transfer Learning Pipeline Complete!")

    return fine_tuned_model, results

In [79]:
# ========================================
# Usage Examples and Testing
# ========================================

def test_single_image_prediction(model, image_path, class_names, transforms):
    """
    Test model on a single image
    """

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transforms(image).unsqueeze(0).to(device)

    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
        _, predicted = torch.max(outputs, 1)

    # Display results
    predicted_class = class_names[predicted.item()]
    confidence = probabilities[predicted.item()].item()

    print(f"Predicted class: {predicted_class}")
    print(f"Confidence: {confidence:.4f}")

    # Show image with prediction
    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.title(f'Prediction: {predicted_class} (Confidence: {confidence:.4f})')
    plt.axis('off')
    plt.show()

    return predicted_class, confidence

In [80]:
def create_quick_demo():
    """
    Quick demo function for testing without full dataset
    """

    print("🎯 Quick Demo - Creating sample data...")

    # Create dummy data for demonstration
    batch_size = 16
    num_classes = 2
    class_names = ['cats', 'dogs']

    # Create dummy datasets
    dummy_train = torch.utils.data.TensorDataset(
        torch.randn(100, 3, 224, 224),
        torch.randint(0, num_classes, (100,))
    )

    dummy_val = torch.utils.data.TensorDataset(
        torch.randn(20, 3, 224, 224),
        torch.randint(0, num_classes, (20,))
    )

    train_loader = DataLoader(dummy_train, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dummy_val, batch_size=batch_size, shuffle=False)

    # Create and train a simple model
    print("🔥 Training ResNet model...")
    model = create_resnet_model(num_classes).to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    # Train for fewer epochs since it's just a demo
    trained_model, history = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        num_epochs=5, device=device
    )

    # Plot results
    plot_training_history(history, "Demo ResNet")

    print("✅ Quick demo complete!")

    return trained_model, history

In [81]:
# ========================================
# Run the Project
# ========================================

# 🌸 OPTION 1: Use Flowers Dataset (RECOMMENDED FOR BEGINNERS)
model, results = run_transfer_learning_pipeline('flowers')

# 🚗 OPTION 2: Use CIFAR-10 Dataset
#model, results = run_transfer_learning_pipeline('cifar10')

# 🐱🐶 OPTION 3: Use Cats vs Dogs (requires Kaggle account)
#model, results = run_transfer_learning_pipeline('cats_dogs')

# 📁 OPTION 4: Manual upload (original behavior)
# model, results = run_transfer_learning_pipeline('manual')

# 🎯 OPTION 5: Quick demo with dummy data
# model, history = create_quick_demo()

# 🔧 OPTION 6: Interactive selection
def interactive_dataset_selection():
    """
    Interactive dataset selection for easy use
    """
    print("🎯 Choose your dataset:")
    print("1. 🌸 Flowers (230MB, 5 classes) - BEST FOR BEGINNERS")
    print("2. 🚗 CIFAR-10 (170MB, 10 classes) - GOOD FOR LEARNING")
    print("3. 🐱🐶 Cats vs Dogs (500MB, 2 classes) - CLASSIC CHOICE")
    print("4. 📁 Manual upload (bring your own images)")
    print("5. 🎯 Quick demo (dummy data, no download)")

    choice = input("Enter your choice (1-5): ").strip()

    if choice == '1':
        print("🌸 Great choice! Flowers dataset is perfect for learning.")
        return run_transfer_learning_pipeline('flowers')
    elif choice == '2':
        print("🚗 CIFAR-10 is excellent for understanding small image classification!")
        return run_transfer_learning_pipeline('cifar10')
    elif choice == '3':
        print("🐱🐶 Classic choice! Make sure you have a Kaggle account.")
        return run_transfer_learning_pipeline('cats_dogs')
    elif choice == '4':
        print("📁 Manual mode - you'll need to upload your own images.")
        return run_transfer_learning_pipeline('manual')
    elif choice == '5':
        print("🎯 Running quick demo with dummy data...")
        return create_quick_demo()
    else:
        print("Invalid choice. Using Flowers dataset as default.")
        return run_transfer_learning_pipeline('flowers')

print("📚 Transfer Learning Project Setup Complete!")
print("\n🚀 QUICK START OPTIONS:")
print("="*50)
print("# Run this for interactive selection:")
print("model, results = interactive_dataset_selection()")
print()
print("# Or directly choose a dataset:")
print("model, results = run_transfer_learning_pipeline('flowers')  # Recommended")
print("model, results = run_transfer_learning_pipeline('cifar10')")
print("model, results = run_transfer_learning_pipeline('cats_dogs')")
print("model, history = create_quick_demo()  # No download needed")
print("\n💡 TIP: Start with 'flowers' dataset - it's perfect for beginners!")

🚀 Starting Transfer Learning Pipeline
📁 Setting up dataset...
🌸 Setting up Flowers dataset (RECOMMENDED)...
🌸 Downloading Flowers dataset...
228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
✅ Flowers dataset downloaded!
Classes found: ['flower_photos']
Location: datasets/flower_photos
📁 Organizing dataset into train/val split...
  flower_photos: 0 train, 0 val
✅ Dataset organized at: flowers_organized
🔄 Creating data transforms...
❌ Could not load dataset: Found no valid file for the classes flower_photos. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp
💡 Make sure the dataset was downloaded correctly.
📚 Transfer Learning Project Setup Complete!

🚀 QUICK START OPTIONS:
# Run this for interactive selection:
model, results = interactive_dataset_selection()

# Or directly choose a dataset:
model, results = run_transfer_learning_pipeline('flowers')  # Recommended
model, results = run_transfer_learning_pipeline('cifar10')
model, results = run_transfer_